In [0]:
%run "/Workspace/utilities"

In [0]:
# COMMAND ----------

dbutils.widgets.text("load_date", "")
dbutils.widgets.dropdown("load_type", "INCREMENTAL", ["INCREMENTAL", "FULL"])

load_date = dbutils.widgets.get("load_date")
load_type = dbutils.widgets.get("load_type")

print(f"Load Date: {load_date}")
print(f"Load Type: {load_type}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Configuration

# COMMAND ----------

# Use config already initialized in utilities
logger = DataLogger("transform_customers")

# Paths - read CSV from source folder
source_path = config.get_source_path("customers.csv")
silver_path = config.get_silver_path("customers")

logger.info(f"Source path: {source_path}")
logger.info(f"Silver path: {silver_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read Source Data (CSV)

# COMMAND ----------

try:
    # Read CSV file from source folder
    df_source = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(source_path)
    
    logger.info(f"Read {df_source.count()} records from source CSV")
    display(df_source.limit(5))
    
except Exception as e:
    logger.error(f"Failed to read source data", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Transformation

# COMMAND ----------

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Clean and standardize data
df_cleaned = df_source \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("phone", when(col("phone").isNull(), "Unknown").otherwise(trim(col("phone")))) \
    .withColumn("city", when(col("city").isNull(), "Unknown").otherwise(trim(col("city")))) \
    .withColumn("country", upper(trim(col("country")))) \
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name"))) \
    .withColumn("email_domain", split(col("email"), "@").getItem(1))

# Remove duplicates based on customer_id
df_deduped = df_cleaned.dropDuplicates(["customer_id"])

logger.info(f"Transformation completed. Removed {df_cleaned.count() - df_deduped.count()} duplicates")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Add Audit Columns

# COMMAND ----------

df_final = df_deduped \
    .withColumn("created_date", current_timestamp()) \
    .withColumn("modified_date", current_timestamp())

logger.info(f"Final record count: {df_final.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write to Silver Layer

# COMMAND ----------

try:
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .save(silver_path)
    
    logger.info(f"✅ Successfully wrote {df_final.count()} records to Silver layer")
    
except Exception as e:
    logger.error("Failed to write to Silver layer", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Summary Statistics

# COMMAND ----------

print(f"\n📊 Transformation Summary:")
print(f"Source records: {df_source.count()}")
print(f"Duplicates removed: {df_cleaned.count() - df_deduped.count()}")
print(f"Final records: {df_final.count()}")

display(df_final.groupBy("country").count().orderBy("count", ascending=False).limit(10))